# PROJECT [CURRENCY CONVERTER USING API]

In [3]:
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
from langchain_core.tools import tool, InjectedToolArg
from langchain_core.messages import HumanMessage, ToolMessage
from typing import Annotated
from dotenv import load_dotenv
import requests
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
@tool
def get_conversion_factor(base_currency:str,target_currency:str)->float:
    """
    This function fetches the currency conversion rate at real time between base currency and target currency
    """
    url = f'https://v6.exchangerate-api.com/v6/e95ce53494bf36351a73e88c/pair/{base_currency}/{target_currency}'
    response=requests.get(url)

    return response.json()

@tool
def converter(base_currency_value:int,conversion_rate: Annotated[float, InjectedToolArg])->float:
    """
    given a currency conversion rate this function calculates the target currency value from a given base currency value
    """
    return base_currency_value*conversion_rate

In [5]:
# =========================
# MODEL
# =========================
llm=HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    task="text-generation"
)

model=ChatHuggingFace(llm=llm)

model_with_tools = model.bind_tools(
    [get_conversion_factor, converter]
)

In [6]:
model_with_tools=model.bind_tools([get_conversion_factor,converter])

In [7]:
messages = [HumanMessage('What is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR')]

In [8]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR', additional_kwargs={}, response_metadata={})]

In [9]:
model_with_tools.invoke(messages)

AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_factor', 'description': None}, 'id': '974e823c6', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 182, 'prompt_tokens': 195, 'total_tokens': 377}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_46358675aa6b1ba4d98d', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e2ae8-0c58-7a30-809b-293e9f9c2cfc-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': '974e823c6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 195, 'output_tokens': 182, 'total_tokens': 377})

In [10]:
ai_msg=model_with_tools.invoke(messages)

In [11]:
ai_msg

AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_factor', 'description': None}, 'id': '9af24916c', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 201, 'prompt_tokens': 195, 'total_tokens': 396}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_46358675aa6b1ba4d98d', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e2ae8-1151-7f11-b0d4-1e023bb4b8d4-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': '9af24916c', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 195, 'output_tokens': 201, 'total_tokens': 396})

In [12]:
ai_msg.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': '9af24916c',
  'type': 'tool_call'}]

In [13]:
tool_call = ai_msg.tool_calls[0]

result = get_conversion_factor.invoke(tool_call["args"])

In [14]:
messages = [
    HumanMessage(
        content="What is conversion factor between INR and USD and convert 10 INR to USD"
    )
]

ai_message = model_with_tools.invoke(messages)

print(ai_message.tool_calls)


[{'name': 'get_conversion_factor', 'args': {'base_currency': 'INR', 'target_currency': 'USD'}, 'id': '1a8417300', 'type': 'tool_call'}]


In [15]:
tool_call = ai_message.tool_calls[0]

result = get_conversion_factor.invoke(tool_call["args"])

print(result)

{'result': 'success', 'documentation': 'https://www.exchangerate-api.com/docs', 'terms_of_use': 'https://www.exchangerate-api.com/terms', 'time_last_update_unix': 1778803201, 'time_last_update_utc': 'Fri, 15 May 2026 00:00:01 +0000', 'time_next_update_unix': 1778889601, 'time_next_update_utc': 'Sat, 16 May 2026 00:00:01 +0000', 'base_code': 'INR', 'target_code': 'USD', 'conversion_rate': 0.01044}


In [16]:
messages.append(ai_message)

messages.append(
    ToolMessage(
        content=str(result),
        tool_call_id=tool_call["id"],
        name=tool_call["name"]
    )
)

In [17]:
ai_message2 = model_with_tools.invoke(messages)

print(ai_message2.tool_calls)

[{'name': 'converter', 'args': {'base_currency_value': 10}, 'id': '64d06865c', 'type': 'tool_call'}]


In [18]:
messages.append(ai_msg)

In [19]:
messages

[HumanMessage(content='What is conversion factor between INR and USD and convert 10 INR to USD', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency":"INR","target_currency":"USD"}', 'name': 'get_conversion_factor', 'description': None}, 'id': '1a8417300', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 188, 'total_tokens': 268}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_46358675aa6b1ba4d98d', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e2ae8-1caf-7971-a8fc-8604af41432e-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'INR', 'target_currency': 'USD'}, 'id': '1a8417300', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 188, 'output_tokens': 80, 'total_tokens': 268}),
 ToolMessage(content="{'result': 'success', 'documentation': 'https://www.exchang

In [25]:
import json

for tool_call in ai_message.tool_calls:

    # execute the 1st tool and get the value of conversion rate
    if tool_call['name'] == 'get_conversion_factor':

        tool_message1 = get_conversion_factor.invoke(tool_call["args"])

        # fetch this conversion rate
        conversion_rate = tool_message1["conversion_rate"]

        # append this tool message to messages list
        messages.append(
            ToolMessage(
                content=json.dumps(tool_message1),
                tool_call_id=tool_call["id"],
                name=tool_call["name"]
                )
            )

    # execute the 2nd tool using the conversion rate from tool 1
    if tool_call['name'] == 'converter':

        # fetch the current arg
        tool_call['args']['conversion_rate'] = conversion_rate

        tool_message2 = converter.invoke(tool_call["args"])

        messages.append(
            ToolMessage(
                content=str(tool_message2),
                tool_call_id=tool_call["id"],
                name=tool_call["name"]
            )
        )

In [26]:
messages

[HumanMessage(content='What is conversion factor between INR and USD and convert 10 INR to USD', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency":"INR","target_currency":"USD"}', 'name': 'get_conversion_factor', 'description': None}, 'id': '1a8417300', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 188, 'total_tokens': 268}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_46358675aa6b1ba4d98d', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e2ae8-1caf-7971-a8fc-8604af41432e-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'INR', 'target_currency': 'USD'}, 'id': '1a8417300', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 188, 'output_tokens': 80, 'total_tokens': 268}),
 ToolMessage(content="{'result': 'success', 'documentation': 'https://www.exchang

In [27]:
model_with_tools.invoke(messages).content

'**Conversion factor (INR\u202f→\u202fUSD)**  \n- 1\u202fINR = **0.01044\u202fUSD**\n\n**10\u202fINR in USD**\n\n\\[\n10\\ \\text{INR} \\times 0.01044\\ \\frac{\\text{USD}}{\\text{INR}} = 0.1044\\ \\text{USD}\n\\]\n\nSo, **10\u202fINR ≈\u202f$0.1044\u202fUSD**.'